# Week 5: финальное решение v6c — ансамбль ResNeXt-101, Wide ResNet-101, ResNet-152 на увеличенных картинках и ResNet-101

**Среда:** Kaggle, ускоритель GPU T4 ×2 (базовая бесплатная настройка), интернет включён.

**Идея.** Шесть предобученных на ImageNet сети семейства ResNet дообучаются с аугментациями под дефекты снимков: ResNeXt-101 (64x4d) и Wide ResNet-101-2 на исходных картинках 256×256 две ResNet-152 на увеличенных до 288×288 и две ResNet-101 на 256×256. Сети разного устройства и разного масштаба ошибаются по-разному, поэтому их усреднение точнее.
Модели делятся поровну между двумя видеокартами и учатся одновременно. На тесте каждая модель смотрит на картинку
один раз, но под своим углом (поворот или отражение), и вероятности усредняются.

* `TRAIN_ON_VAL = False` — модели учатся на `train`, точность каждой и ансамбля меряется на `val` (для отчёта).
* `TRAIN_ON_VAL = True` — финал: модели учатся на `train + val` и предсказывают тест.

Предсказания сохраняются в `week5_solution_v6c.csv` (имя как у ноутбука) и в `final_submission.csv`.

In [1]:
NOTEBOOK_NAME = "week5_solution_v6c"
TRAIN_ON_VAL = True                               # False: проверка на val. True: финальное обучение на train + val

BASE = {                                          # общие настройки всех моделей
    "arch": "resnet152",
    "weights": "IMAGENET1K_V2",                   # веса ImageNet из torchvision
    "unfreeze": ["layer2", "layer3", "layer4"],   # какие блоки дообучаются
    "dropout": 0.3,
    "img_size": 256,                              # исходный размер снимков, без уменьшения
    "batch_size": 64,
    "epochs": 10,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "p_blur": 0.3, "p_lowres": 0.15, "p_gray": 0.15, "p_dark": 0.15, "p_noise": 0.15,   # аугментации под дефекты
    "augmix": False,
    "seed": 42,
}

# Модели ансамбля: чем каждая отличается от BASE и под каким видом смотрит на тест.
# Вид 0..3 - поворот на 0/90/180/270 градусов, 4..7 - то же с отражением.
# ResNeXt и Wide ResNet - разновидности ResNet из того же модуля torchvision (построены на классе ResNet),
# их использование подтверждено преподавателем. У resnext101_64x4d в torchvision есть только веса IMAGENET1K_V1.
RX64 = {"arch": "resnext101_64x4d", "weights": "IMAGENET1K_V1", "batch_size": 32}
RX32 = {"arch": "resnext101_32x8d", "batch_size": 32}
WRN = {"arch": "wide_resnet101_2", "batch_size": 32}
MEMBERS = [
    {**RX64, "seed": 42, "view": 0},
    {**WRN, "seed": 42, "view": 4},
    {"arch": "resnet152", "img_size": 288, "batch_size": 48, "seed": 42, "view": 1},
    {"arch": "resnet152", "img_size": 288, "batch_size": 48, "seed": 1, "view": 5},
    {"arch": "resnet101", "seed": 42, "view": 2},
    {"arch": "resnet101", "seed": 1, "view": 6},
]

In [2]:
import threading
import copy, random, time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"                        # смешанная точность: на GPU обучение примерно вдвое быстрее
NUM_WORKERS = 2 if DEVICE == "cuda" else 0        # аугментации в отдельных процессах, чтобы GPU не простаивал
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]   # на Kaggle T4 x2 это две видеокарты

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

COMPETITION = "week-5-competition-geo-satellites-classification"

def find_data():
    """Папка с train.csv: свой компьютер, Colab, входные данные Kaggle."""
    for folder in (Path(COMPETITION), Path("."), Path("/content"), Path("data")):
        if (folder / "train.csv").exists():
            return folder
    found = list(Path("/kaggle/input").rglob("train.csv")) if Path("/kaggle/input").exists() else []
    if found:
        return found[0].parent
    try:                                            # ноутбук Kaggle: данные соревнования через kagglehub
        import kagglehub
        return next(Path(kagglehub.competition_download(COMPETITION)).rglob("train.csv")).parent
    except Exception as error:
        print("kagglehub failed:", error)
    # В запуске на Kaggle здесь скачивался архив с данными по ссылке из бейзлайна курса, поэтому в выводе ниже путь /kaggle/temp/data.
    # Данные соревнования закрытые, поэтому в версии для репозитория ссылка убрана.
    raise FileNotFoundError(f"Положите данные соревнования в папку {COMPETITION}/ рядом с ноутбуком")

DATA_DIR = find_data()

tables = {name: pd.read_csv(DATA_DIR / f"{name}.csv", dtype={"id": str}) for name in ("train", "val", "test")}
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
NUM_CLASSES = len(classes)

print("devices:", GPUS, "| data:", DATA_DIR)
print({name: len(table) for name, table in tables.items()}, "| classes:", NUM_CLASSES)

kagglehub failed: POST failed with: {"errors":["New Competitions cannot be attached in non-interactive sessions."],"error":{"code":9},"wasSuccessful":false}
devices: ['cuda:0', 'cuda:1'] | data: /kaggle/temp/data
{'train': 4999, 'val': 699, 'test': 8299} | classes: 20


## Данные и аугментации

In [3]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class LowRes:
    """Низкое разрешение: уменьшить картинку в 2-4 раза и растянуть обратно."""
    def __call__(self, img):
        small = img.size[0] // int(torch.randint(2, 5, (1,)))
        return img.resize((small, small), Image.BILINEAR).resize(img.size, Image.BILINEAR)

class AddNoise:
    """Шум: к каждому числу тензора прибавить случайную добавку."""
    def __call__(self, x):
        strength = float(torch.empty(1).uniform_(0.03, 0.12))
        return (x + torch.randn_like(x) * strength).clamp(0, 1)

def make_train_transform(cfg):
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([transforms.RandomRotation((angle, angle)) for angle in (0, 90, 180, 270)]),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        *([transforms.AugMix()] if cfg["augmix"] else []),
        transforms.RandomApply([transforms.ColorJitter(brightness=(0.3, 0.7))], p=cfg["p_dark"]),
        transforms.RandomGrayscale(p=cfg["p_gray"]),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=9, sigma=(0.5, 3.0))], p=cfg["p_blur"]),
        transforms.RandomApply([LowRes()], p=cfg["p_lowres"]),
        transforms.ToTensor(),
        transforms.RandomApply([AddNoise()], p=cfg["p_noise"]),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class ImageDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]

IMAGE_CACHE = {}
def load_images(folder, size):
    """Картинки папки, уменьшенные до size. Читаются один раз для каждого размера и остаются в памяти."""
    if (folder, size) not in IMAGE_CACHE:
        start = time.time()
        method = Image.BOX if size <= 256 else Image.BICUBIC      # уменьшаем усреднением, увеличиваем бикубически
        IMAGE_CACHE[(folder, size)] = [Image.open(DATA_DIR / folder / image_id).convert("RGB").resize((size, size), method)
                                       for image_id in tables[folder].id]
        print(f"  loaded {folder} at {size}px in {time.time() - start:.0f}s")
    return IMAGE_CACHE[(folder, size)]

def make_loader(dataset, cfg, shuffle):
    generator = torch.Generator().manual_seed(cfg["seed"])
    return DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=shuffle, generator=generator, num_workers=NUM_WORKERS)

## Модель и обучение
Каждой модели назначается своя видеокарта, поэтому устройство передаётся в функции явно.

In [4]:
def build_model(cfg):
    """Предобученная ResNet: заморожено всё, кроме блоков из cfg["unfreeze"]; свой классификатор на 20 классов."""
    seed_everything(cfg["seed"])
    backbone = getattr(models, cfg["arch"])(weights=cfg["weights"])
    unfreeze_all = "all" in cfg["unfreeze"]
    for name, param in backbone.named_parameters():
        param.requires_grad = unfreeze_all or name.split(".")[0] in cfg["unfreeze"]
    num_features = backbone.fc.in_features
    backbone.fc = nn.Identity()
    classifier = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(num_features, NUM_CLASSES))
    return nn.Sequential(backbone, classifier)

def train(model, train_data, cfg, device):
    use_amp = device.startswith("cuda")
    if use_amp:
        with torch.cuda.device(device):
            torch.cuda.manual_seed(cfg["seed"])          # сид случайности на этой видеокарте (Dropout)
    model.to(device)
    loader = make_loader(train_data, cfg, shuffle=True)   # порядок картинок и аугментации задаются сидом загрузчика
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg["lr"], total_steps=cfg["epochs"] * len(loader))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for epoch in range(cfg["epochs"]):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

def to_view(images, view):
    """Вид картинки: поворот на 90 градусов view % 4 раз и, для view >= 4, отражение."""
    images = torch.rot90(images, view % 4, (2, 3))
    return images.flip(3) if view >= 4 else images

@torch.no_grad()
def predict_probs(model, images_u8, cfg, device):
    """Вероятности классов. images_u8 - все картинки одним тензором (N, H, W, 3) uint8; нормализация делается на GPU."""
    use_amp = device.startswith("cuda")
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1) * 255
    std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1) * 255
    model.eval()
    outputs = []
    for start in range(0, len(images_u8), 128):
        batch = images_u8[start:start + 128].to(device).permute(0, 3, 1, 2).float()
        batch = to_view((batch - mean) / std, cfg["view"])
        with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
            outputs.append(model(batch).float().softmax(dim=1).cpu())
    return torch.cat(outputs)

def run_on_all_gpus(job, items):
    """Делит items между видеокартами по очереди и выполняет job(item, device) на каждой карте в своём потоке."""
    errors = []
    def worker(device, part):
        try:
            for item in part:
                job(item, device)
        except Exception as error:
            errors.append(error)
    threads = [threading.Thread(target=worker, args=(device, items[i::len(GPUS)])) for i, device in enumerate(GPUS)]
    for t in threads: t.start()
    for t in threads: t.join()
    if errors:
        raise errors[0]

def describe(changes):
    return ", ".join(f"{k}={v}" for k, v in changes.items())

## Обучение ансамбля
Сначала все модели создаются по очереди (так начальные веса классификатора зависят только от сида модели),
потом учатся параллельно: каждая видеокарта обучает свою половину списка.

In [5]:
notebook_start = time.time()
as_tensor = lambda images: torch.from_numpy(np.stack([np.asarray(img) for img in images]))   # (N, H, W, 3) uint8
sizes = sorted({{**BASE, **changes}["img_size"] for changes in MEMBERS})       # размеры картинок, которые нужны моделям
test_u8, val_u8 = {}, {}
for size in sizes:
    test_u8[size] = as_tensor(load_images("test", size))
    IMAGE_CACHE.pop(("test", size))                 # тест нужен только тензором: освобождаем память
    val_u8[size] = as_tensor(load_images("val", size))
val_labels = torch.tensor(tables["val"].label.tolist())

ensemble = []
for changes in MEMBERS:
    cfg = {**BASE, **changes}
    size = cfg["img_size"]
    transform = make_train_transform(cfg)
    train_data = ImageDataset(load_images("train", size), tables["train"].label.tolist(), transform)
    if TRAIN_ON_VAL:
        train_data = ConcatDataset([train_data, ImageDataset(load_images("val", size), tables["val"].label.tolist(), transform)])
    ensemble.append({"name": describe(changes), "cfg": cfg, "model": build_model(cfg), "train_data": train_data})

def train_member(member, device):
    start = time.time()
    train(member["model"], member["train_data"], member["cfg"], device)
    member["device"], member["train_minutes"] = device, (time.time() - start) / 60
    print(f"trained {member['name']} on {device} in {member['train_minutes']:.1f} min", flush=True)

# Замер времени обучения финального решения (лимит по правилам: не более 45 минут)
start_time = time.time()
run_on_all_gpus(train_member, ensemble)
end_time = time.time()
elapsed_minutes = (end_time - start_time) / 60

print(f"\nElapsed time: {elapsed_minutes:.2f} minutes")

  loaded test at 256px in 7s
  loaded val at 256px in 0s
  loaded test at 288px in 19s
  loaded val at 288px in 2s
  loaded train at 256px in 4s
Downloading: "https://download.pytorch.org/models/resnext101_64x4d-173b62eb.pth" to /root/.cache/torch/hub/checkpoints/resnext101_64x4d-173b62eb.pth


100%|██████████| 319M/319M [00:01<00:00, 194MB/s]


Downloading: "https://download.pytorch.org/models/wide_resnet101_2-d733dc28.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet101_2-d733dc28.pth


100%|██████████| 485M/485M [00:01<00:00, 255MB/s]


  loaded train at 288px in 11s
Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:01<00:00, 228MB/s]


Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 261MB/s]


trained arch=wide_resnet101_2, batch_size=32, seed=42, view=4 on cuda:1 in 13.8 min
trained arch=resnext101_64x4d, weights=IMAGENET1K_V1, batch_size=32, seed=42, view=0 on cuda:0 in 15.8 min
trained arch=resnet152, img_size=288, batch_size=48, seed=1, view=5 on cuda:1 in 12.9 min
trained arch=resnet152, img_size=288, batch_size=48, seed=42, view=1 on cuda:0 in 14.5 min
trained arch=resnet101, seed=1, view=6 on cuda:1 in 7.1 min
trained arch=resnet101, seed=42, view=2 on cuda:0 in 7.7 min

Elapsed time: 38.02 minutes


## Проверка на отложенной валидации
Считается только при `TRAIN_ON_VAL = False`: в этом режиме модели не видели `val` при обучении.

In [6]:
rows = []
if not TRAIN_ON_VAL:
    def predict_val(member, device):
        member["val_probs"] = predict_probs(member["model"], val_u8[member["cfg"]["img_size"]], member["cfg"], member["device"])
    run_on_all_gpus(predict_val, ensemble)
    for member in ensemble:
        acc = (member["val_probs"].argmax(dim=1) == val_labels).float().mean().item()
        rows.append({"model": member["name"], "device": member["device"], "train_minutes": round(member["train_minutes"], 1), "val_acc": round(acc, 4)})
    ensemble_acc = (sum(m["val_probs"] for m in ensemble).argmax(dim=1) == val_labels).float().mean().item()
    rows.append({"model": "ENSEMBLE", "device": " + ".join(GPUS), "train_minutes": round(sum(m["train_minutes"] for m in ensemble), 1), "val_acc": round(ensemble_acc, 4)})
    print(f"ENSEMBLE validation accuracy: {ensemble_acc:.4f}")
else:
    rows = [{"model": m["name"], "device": m["device"], "train_minutes": round(m["train_minutes"], 1), "val_acc": float("nan")} for m in ensemble]
print(pd.DataFrame(rows).to_string(index=False))
pd.DataFrame(rows).assign(seed=[m["cfg"]["seed"] for m in ensemble] + ([""] if not TRAIN_ON_VAL else []),
                          trained_on="train+val" if TRAIN_ON_VAL else "train").to_csv(
    f"{NOTEBOOK_NAME}_{'final' if TRAIN_ON_VAL else 'validation'}_log.csv", index=False)

                                                                       model device  train_minutes  val_acc
arch=resnext101_64x4d, weights=IMAGENET1K_V1, batch_size=32, seed=42, view=0 cuda:0           15.8      NaN
                       arch=wide_resnet101_2, batch_size=32, seed=42, view=4 cuda:1           13.8      NaN
                arch=resnet152, img_size=288, batch_size=48, seed=42, view=1 cuda:0           14.5      NaN
                 arch=resnet152, img_size=288, batch_size=48, seed=1, view=5 cuda:1           12.9      NaN
                                             arch=resnet101, seed=42, view=2 cuda:0            7.7      NaN
                                              arch=resnet101, seed=1, view=6 cuda:1            7.1      NaN


## Предсказание теста
Каждая модель один раз проходит по тесту на своей видеокарте, вероятности складываются, ответ — класс с наибольшей суммой.

In [7]:
def predict_test(member, device):
    member["test_probs"] = predict_probs(member["model"], test_u8[member["cfg"]["img_size"]], member["cfg"], member["device"])

start = time.time()
run_on_all_gpus(predict_test, ensemble)
labels = sum(m["test_probs"] for m in ensemble).argmax(dim=1).numpy()
print(f"test prediction time: {time.time() - start:.0f}s | whole notebook: {(time.time() - notebook_start) / 60:.1f} min")

submission = pd.DataFrame({"id": tables["test"].id.tolist(), "label": labels})
assert list(submission.columns) == ["id", "label"]
assert len(submission) == len(tables["test"]) and submission.label.between(0, NUM_CLASSES - 1).all()

file_name = f"{NOTEBOOK_NAME}.csv" if TRAIN_ON_VAL else f"{NOTEBOOK_NAME}_trained_without_val.csv"
submission.to_csv(file_name, index=False)
if TRAIN_ON_VAL:
    submission.to_csv("final_submission.csv", index=False)      # то же самое под именем, которое требуют правила
print("saved:", file_name, "| rows:", len(submission))
submission.head()

test prediction time: 127s | whole notebook: 41.2 min
saved: week5_solution_v6c.csv | rows: 8299


,id,label
0,2.jpg,4
1,4.jpg,5
2,5.jpg,11
3,7.jpg,18
4,8.jpg,10
